In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter, sobel

class GaussianNaiveBayesScratch:
    def __init__(self, var_smoothing=1e-4):
        self.var_smoothing = var_smoothing
        self.classes, self.priors = None, None
        self.means, self.vars = None, None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        self.priors = np.zeros(n_classes)
        self.means = np.zeros((n_classes, n_features))
        self.vars = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.priors[idx] = X_c.shape[0] / float(n_samples)
            self.means[idx, :] = np.mean(X_c, axis=0)
            self.vars[idx, :] = np.var(X_c, axis=0) + self.var_smoothing
        return self

    def _calc_log_likelihood(self, class_idx, X):
        m, v = self.means[class_idx], self.vars[class_idx]
        lp = np.log(self.priors[class_idx])
        t1 = -0.5 * np.sum(np.log(2.0 * np.pi * v))
        t2 = -0.5 * np.sum(((X - m)**2) / v, axis=1)
        return lp + t1 + t2

    def predict_proba(self, X):
        ll = np.array([self._calc_log_likelihood(i, X)
                       for i in range(len(self.classes))]).T
        m = np.max(ll, axis=1, keepdims=True)
        lse = m + np.log(np.sum(np.exp(ll - m), axis=1, keepdims=True))
        return np.exp(ll - lse)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

# Feature Extraction (Sobel + 3x3 Texture Statistics)
def extract_pixel_features(gray):
    gx = sobel(gray, axis=1); gy = sobel(gray, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    local_mean = uniform_filter(gray, size=3)
    local_sq = uniform_filter(gray**2, size=3)
    local_std = np.sqrt(np.clip(local_sq - local_mean**2, 0, None))
    return np.stack([gray, grad_mag, gx, gy, local_mean, local_std], axis=-1)

# Evaluation Metrics
def classification_report(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*prec*rec / (prec + rec) if (prec + rec) else 0.0
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
